In [0]:
dbutils.widgets.text("target_schema", "5_projects.pdac003")
dbutils.widgets.dropdown("require_refreshed_cohort", "true", ["true", "false"])

TARGET = dbutils.widgets.get("target_schema")
REQUIRE_REFRESHED_COHORT = dbutils.widgets.get("require_refreshed_cohort") == "true"

COHORT_TABLE = "`6_mgmt`.cohorts.pdac003"

# IG cut-off, unchanged from v1: a column is dropped if either tag is above these values.
MAX_IG_RISK = 5
MAX_IG_SEVERITY = 5
ALWAYS_EXCLUDED_COLUMNS = {"adc_updt"}  # pipeline load timestamp, compared case-insensitively

target_catalog, target_schema = TARGET.split(".")
assert target_catalog in ("5_projects", "8_dev"), f"Unexpected target catalog: {TARGET}"
print(f"Building PDAC003 views in {TARGET}")

## 1. Cohort
Every view filters on the live cohort table, so re-running the cohort refresh changes the views without rebuilding them.
(The bloodcount sample bridge in step 4 is the exception: it is a table and is refreshed by re-running this notebook.)

In [0]:
cohort_columns = [c.lower() for c in spark.table(COHORT_TABLE).columns]
if REQUIRE_REFRESHED_COHORT:
    assert "person_id_source" in cohort_columns, (
        f"{COHORT_TABLE} has no PERSON_ID_SOURCE column, so the 2026-11-17 cohort refresh has not been applied. "
        "Run CC/PDAC003_Cohort_Refresh_20261117 with apply = true first."
    )


def cohort_person_ids(as_type="BIGINT"):
    """SQL subquery returning the cohort's PERSON_IDs, cast to the type of the column it is compared with."""
    return f"SELECT DISTINCT CAST(PERSON_ID AS {as_type}) FROM {COHORT_TABLE} WHERE PERSON_ID IS NOT NULL"


cohort_size = spark.sql(f"""
    SELECT count(DISTINCT NHS_NUMBER) AS nhs_numbers, count(DISTINCT PERSON_ID) AS persons FROM {COHORT_TABLE}
""").collect()[0]
print(f"Cohort: {cohort_size.nhs_numbers:,} NHS numbers, {cohort_size.persons:,} with a PERSON_ID")

## 2. Shared helpers: IG column exclusion and source metadata
The tags and comments for a whole source schema are read once, then reused for its views and for the schema mapping.

In [0]:
def read_source_metadata(catalog, schema):
    """Return {table: {column_lower: info}} for one source schema.

    info holds the column's real name, position, type, comment and IG tags. One query covers the whole schema.
    """
    rows = spark.sql(f"""
        WITH tags AS (
            SELECT table_name, column_name,
                   max(CASE WHEN tag_name = 'ig_risk'     THEN tag_value END) AS ig_risk,
                   max(CASE WHEN tag_name = 'ig_severity' THEN tag_value END) AS ig_severity
            FROM `{catalog}`.information_schema.column_tags
            WHERE schema_name = '{schema}' AND tag_name IN ('ig_risk', 'ig_severity')
            GROUP BY table_name, column_name
        )
        SELECT c.table_name, c.column_name, c.ordinal_position, c.full_data_type, c.comment,
               t.ig_risk, t.ig_severity
        FROM `{catalog}`.information_schema.columns c
        LEFT JOIN tags t ON t.table_name = c.table_name AND t.column_name = c.column_name
        WHERE c.table_schema = '{schema}'
    """).collect()
    metadata = {}
    for r in rows:
        metadata.setdefault(r.table_name, {})[r.column_name.lower()] = r.asDict()
    return metadata


def is_excluded(info):
    """True if the IG cut-off or the always-excluded list removes this column."""
    def above(value, limit):
        return value is not None and int(value) > limit
    return (
        info["column_name"].lower() in ALWAYS_EXCLUDED_COLUMNS
        or above(info["ig_risk"], MAX_IG_RISK)
        or above(info["ig_severity"], MAX_IG_SEVERITY)
    )


def delivered_columns(table_metadata):
    """Source columns kept for delivery, in source order (column info dicts)."""
    kept = [info for info in table_metadata.values() if not is_excluded(info)]
    return sorted(kept, key=lambda info: info["ordinal_position"])


# ADF's Copy activity can't read Parquet ARRAY, STRUCT or MAP columns (only Data Flows can), so columns of these
# types are delivered as JSON text. Today that is 12 ARRAY columns in gold clinical_v_imaging_accession and
# 3 STRUCT columns in gold clinical_registry_surgery_episode.
NESTED_TYPE_PREFIXES = ("array", "struct", "map", "variant")


def is_nested(info):
    """True if the column's type is one ADF Copy can't handle, so it is delivered as JSON text."""
    return info["full_data_type"].lower().startswith(NESTED_TYPE_PREFIXES)


def delivered_type(info):
    """The column's type as delivered, for the schema mapping."""
    if is_nested(info):
        return f"string (JSON of {info['full_data_type']})"
    return info["full_data_type"]


def select_list(columns, alias):
    """Comma-separated, back-quoted column list prefixed with a table alias.

    Nested columns are wrapped in to_json() and keep their name.
    """
    def one_column(info):
        column = f"{alias}.`{info['column_name']}`"
        if is_nested(info):
            return f"to_json({column}) AS `{info['column_name']}`"
        return column
    return ", ".join(one_column(info) for info in columns)


# Every view built below is recorded here; the schema mapping in step 6 is generated from this list.
# Each entry: dataset, view name, source table, delivered columns, extra columns, cohort filter description.
built_views = []

## 3. Clear the old views
Drops every view in the target schema, so renamed or retired views (v1's unprefixed bloodcount views and `schema`)
don't get delivered by ADF.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET}")


def views_in_target():
    """Names of the views currently in the target schema.

    information_schema is used because SHOW VIEWS rejects a catalog-qualified schema on this runtime.
    """
    return {
        r.table_name for r in spark.sql(f"""
            SELECT table_name FROM `{target_catalog}`.information_schema.tables
            WHERE table_schema = '{target_schema}' AND table_type = 'VIEW'
        """).collect()
    }


old_views = sorted(views_in_target())
for view in old_views:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET}.`{view}`")
print(f"Dropped {len(old_views)} existing views")

## 4. RDE
Same 46 tables as v1, filtered on whichever person column the table uses.

In [0]:
RDE_TABLES = [
    "rde_aliases", "rde_all_procedures", "rde_allergydetails", "rde_apc_diagnosis", "rde_apc_opcs",
    "rde_ariapharmacy", "rde_blobdataset", "rde_cds_apc", "rde_cds_opa", "rde_critactivity", "rde_critopcs",
    "rde_critperiod", "rde_emergencyd", "rde_emergency_findings", "rde_encounter", "rde_family_history",
    "rde_iqemo", "rde_mat_nnu_episodes", "rde_mat_nnu_exam", "rde_mat_nnu_nccmds", "rde_measurements",
    "rde_medadmin", "rde_mill_powertrials", "rde_msds_booking", "rde_msds_carecontact", "rde_msds_delivery",
    "rde_msds_diagnosis", "rde_op_diagnosis", "rde_opa_opcs", "rde_pathology", "rde_patient_demographics",
    "rde_pc_diagnosis", "rde_pc_problems", "rde_pc_procedures", "rde_pharmacyorders", "rde_powerforms",
    "rde_radiology", "rde_raw_pathology", "rde_scr_careplan", "rde_scr_deftreatment", "rde_scr_demographics",
    "rde_scr_diagnosis", "rde_scr_investigations", "rde_scr_pathology", "rde_scr_referrals",
    "rde_scr_trackingcomments",
]

rde_metadata = read_source_metadata("4_prod", "rde")


def rde_person_column(table_metadata):
    """The table's person column info (RDE tables spell it PERSON_ID, PERSONID or Person_ID)."""
    for name in ("person_id", "personid"):
        if name in table_metadata:
            return table_metadata[name]
    return None


for table in RDE_TABLES:
    table_metadata = rde_metadata.get(table)
    assert table_metadata, f"4_prod.rde.{table} no longer exists"
    person = rde_person_column(table_metadata)
    # v1 published a table with no person column unfiltered (the whole Trust). Refuse that instead.
    assert person, f"4_prod.rde.{table} has no person column, so it can't be restricted to the cohort"

    columns = delivered_columns(table_metadata)
    spark.sql(f"""
        CREATE OR REPLACE VIEW {TARGET}.`{table}` AS
        SELECT {select_list(columns, 's')}
        FROM `4_prod`.rde.`{table}` s
        WHERE s.`{person['column_name']}` IN ({cohort_person_ids(person['full_data_type'])})
    """)
    built_views.append(dict(dataset="rde", view=table, source=f"4_prod.rde.{table}", columns=columns,
                            extra_columns=[], cohort_filter=f"{person['column_name']} in cohort"))

print(f"RDE: {len(RDE_TABLES)} views")

## 5. Bloodcounts
Bloodcount tables carry an analyser sample number, not a person. It matches `rde_raw_pathology.LabNo` (measured
2026-11-17: about 95% of samples match, and 99% of matched LabNos belong to one person).

**Step 5a** materialises a small bridge table with one row per cohort LabNo. It is a table rather than a view,
so the 66 views don't each scan the 1.4B-row pathology table. The deliver job exports views only, so the bridge
itself is not delivered. A LabNo held by more than one person is left out.

In [0]:
BRIDGE = f"{TARGET}.bloodcounts_sample_person_bridge"

spark.sql(f"""
CREATE OR REPLACE TABLE {BRIDGE}
COMMENT 'PDAC003: cohort LabNo -> PERSON_ID, one row per LabNo, used by the bloodcounts_* views. LabNos held by more than one person are left out. Rebuilt by PDAC003_v2.'
AS
WITH
-- LabNos belonging to at least one cohort member.
cohort_labno AS (
    SELECT DISTINCT LabNo
    FROM `4_prod`.rde.rde_raw_pathology
    WHERE LabNo IS NOT NULL
      AND PERSON_ID IN ({cohort_person_ids('BIGINT')})
),
-- Every person each of those LabNos is recorded against, cohort or not.
labno_people AS (
    SELECT p.LabNo, count(DISTINCT p.PERSON_ID) AS people, min(p.PERSON_ID) AS PERSON_ID
    FROM `4_prod`.rde.rde_raw_pathology p
    WHERE p.LabNo IN (SELECT LabNo FROM cohort_labno)
    GROUP BY p.LabNo
)
SELECT LabNo AS sample_no, PERSON_ID
FROM labno_people
WHERE people = 1
""")

bridge_check = spark.sql(f"SELECT count(*) AS n, count(DISTINCT sample_no) AS samples, count(DISTINCT PERSON_ID) AS persons FROM {BRIDGE}").collect()[0]
assert bridge_check.n == bridge_check.samples, "Bridge has repeated sample numbers: the bloodcount views would repeat rows"
print(f"Bridge: {bridge_check.samples:,} LabNos for {bridge_check.persons:,} cohort persons")

**Step 5b:** one view per bloodcount table, with `PERSON_ID` as the first column. Sample numbers are trimmed
before matching, because a few source values have padding spaces.

In [0]:
BLOODCOUNT_TABLES = [
    "action_message", "ana_internal1", "ana_internal2", "bb", "bb_f", "cal", "cls_param", "dst_data", "dst_data_2",
    "err", "flag_sus2", "flagging", "flg_gate", "giveup", "hsa", "hsa_f", "ig", "ipu_csv1", "ipu_csv2", "jo_aflas",
    "jo_aflas01", "jo_aflas03", "jo_aflas04", "lyact", "mix_issue", "mt_data", "neonew_hpc", "neuteoissue",
    "otherinfo", "outputdata", "plt_abn_sus", "plt_clumps_wnr_gate", "plt_clumps_wnr_v17_18", "plt_clumps_wnr_v21",
    "plt_dst_raw_data", "plt_swt", "rbc_abn_sus", "rbc_dst_raw_data", "reportable", "reportable_f", "research",
    "research_f", "sampling_plt", "sampling_pltf", "sampling_rbc", "sampling_ret", "sampling_wdf",
    "sampling_wdf_2times", "sampling_wnr", "sampling_wpc", "sampling_wpc_2times", "sampling_wpc_3times",
    "sampling_wpc_4times", "sct_ret", "sct_wdf", "sct_wnr", "service_in", "service_out", "servicesettinglog",
    "t_data", "wbc_abn_sct_bf", "wbc_abn_sus", "wdf_low_sfl", "wnr_aggl", "wp_reana", "xn_sample",
]

bloodcount_metadata = read_source_metadata("4_prod", "ancil_bloodcounts")


def sample_number_column(table_metadata):
    """The column holding the analyser sample number (spelt three ways across the bloodcount tables)."""
    for name in ("sample_no_", "sample_no", "sampleid"):
        if name in table_metadata:
            return table_metadata[name]["column_name"]
    return None


PERSON_ID_EXTRA = dict(column_name="PERSON_ID", full_data_type="bigint", source_column="rde_raw_pathology.PERSON_ID",
                       comment="Millennium PERSON_ID, resolved from the sample number via rde_raw_pathology.LabNo (added for this delivery)")

for table in BLOODCOUNT_TABLES:
    table_metadata = bloodcount_metadata.get(table)
    assert table_metadata, f"4_prod.ancil_bloodcounts.{table} no longer exists"
    assert "person_id" not in table_metadata, f"{table} already has a PERSON_ID column; the added one would clash"
    sample_col = sample_number_column(table_metadata)
    assert sample_col, f"{table} has no sample number column"

    columns = delivered_columns(table_metadata)
    view = f"bloodcounts_{table}"
    spark.sql(f"""
        CREATE OR REPLACE VIEW {TARGET}.`{view}` AS
        SELECT p.PERSON_ID, {select_list(columns, 'b')}
        FROM `4_prod`.ancil_bloodcounts.`{table}` b
        INNER JOIN {BRIDGE} p ON trim(b.`{sample_col}`) = p.sample_no
    """)
    built_views.append(dict(dataset="bloodcounts", view=view, source=f"4_prod.ancil_bloodcounts.{table}",
                            columns=columns, extra_columns=[PERSON_ID_EXTRA],
                            cohort_filter=f"trim({sample_col}) = rde_raw_pathology.LabNo of exactly one cohort person"))

print(f"Bloodcounts: {len(BLOODCOUNT_TABLES)} views")

## 6. Gold Journey
Every clinical, spine, reference and text table in `4_prod.gold`, except `text_safety_document`. Each table falls into one of these classes:
1. **Has `person_id`** (or `parent_person_id`): filter on it directly.
2. **Child table without a person column:** keep rows whose key appears in a cohort row of the parent table.
   Links checked 2026-11-17: 99.7 to 100% of child rows match a parent.
3. **Not patient-level** (lookups, locations, practitioners, ward bed state): delivered in full.

A gold table that fits none of these stops the run, so a new gold table is a deliberate decision, never a silent leak or loss.

In [0]:
GOLD_EXCLUDED = {
    "text_safety_document",  # excluded by request, 2026-11-17
}

# Class 2: child table -> (child key columns, [(parent table, parent key columns), ...]).
# A row is kept if its key matches the key of any cohort row in any listed parent.
GOLD_CHILD_LINKS = {
    "clinical_appointment_booking":              (["sch_event_id"],      [("clinical_appointment", ["sch_event_id"])]),
    "clinical_appointment_resource":             (["sch_event_id"],      [("clinical_appointment", ["sch_event_id"])]),
    "clinical_condition_revision":               (["problem_id"],        [("clinical_condition", ["problem_id"])]),
    "clinical_hrg_grouping_diagnosis":           (["cds_id"],            [("clinical_hrg_grouping", ["cds_id"])]),
    "clinical_hrg_grouping_procedure":           (["cds_id"],            [("clinical_hrg_grouping", ["cds_id"])]),
    "clinical_medication_order_ingredient":      (["order_id"],          [("clinical_medication_order", ["order_id"])]),
    "clinical_neonatal_care_day_activity":       (["entity_id"],         [("clinical_neonatal_care_day", ["entity_id"])]),
    "clinical_neonatal_care_day_high_cost_drug": (["entity_id"],         [("clinical_neonatal_care_day", ["entity_id"])]),
    "clinical_procedure_implant_attribute":      (["implant_event_id"],  [("clinical_procedure", ["implant_event_id"])]),
    "clinical_rtt_pathway_encounter_type":       (["pathway_oid"],       [("clinical_rtt_pathway", ["pathway_oid"])]),
    # Safety incidents: incident-level rows for every incident a cohort member took part in.
    "clinical_safety_incident_action":           (["inc_id"],            [("clinical_safety_incident_participant", ["inc_id"])]),
    "clinical_safety_incident_factor":           (["inc_id"],            [("clinical_safety_incident_participant", ["inc_id"])]),
    "clinical_safety_incident_injury":           (["inc_id"],            [("clinical_safety_incident_participant", ["inc_id"])]),
    "clinical_safety_incident_lfpse":            (["inc_id"],            [("clinical_safety_incident_participant", ["inc_id"])]),
    "clinical_safety_incident_status":           (["inc_id"],            [("clinical_safety_incident_participant", ["inc_id"])]),
    "spine_encounter_identifier":                (["encounter_id"],      [("spine_encounter", ["encounter_id"])]),
    "spine_episode_encounter":                   (["episode_id"],        [("spine_episode", ["episode_id"])]),
    "spine_journey_link":                        (["journey_key"],       [("spine_journey", ["journey_key"])]),
    "text_document_sensitivity_label":           (["patient_event_key"], [("text_document", ["patient_event_key"])]),
    # Reference tables that are patient-level despite the prefix.
    "reference_cost_line_item":            (["extract_cd", "activity_record_id"], [("clinical_costed_activity", ["extract_cd", "activity_record_id"])]),
    "reference_elective_access_procedure": (["waiting_list_oid"],  [("clinical_elective_access_entry", ["waiting_list_oid"])]),
    "reference_encounter_attribute":       (["encounter_id"],      [("spine_encounter", ["encounter_id"])]),
    "reference_encounter_bounds":          (["encounter_id"],      [("spine_encounter", ["encounter_id"])]),
    "reference_gene_tested":               (["genomic_test_key"],  [("clinical_genomic_result", ["genomic_test_key"])]),
    "reference_pregnancy_reconciliation":  (["pregnancy_id"],      [("clinical_maternity_care_contact", ["pregnancy_id"]),
                                                                     ("clinical_labour_delivery", ["pregnancy_id"]),
                                                                     ("clinical_baby_delivery", ["pregnancy_id"])]),
}

# Class 3: not patient-level, delivered in full.
GOLD_FULL_TABLES = {
    "reference_admission_metrics", "reference_area_census2021_oa", "reference_area_iod2025_lsoa",
    "reference_concept_map", "reference_concept_registry", "reference_location", "reference_location_attribute_history",
    "reference_location_unit", "reference_organization", "reference_practitioner", "reference_practitioner_group",
    "reference_practitioner_identifier", "reference_practitioner_location_evidence", "reference_research_study",
    "reference_service", "reference_value_set",
    "clinical_bed_status_observation", "clinical_bed_status_observation_coverage",  # ward bed state, no patient
}

gold_metadata = read_source_metadata("4_prod", "gold")
gold_tables = sorted(
    r.table_name for r in spark.sql("""
        SELECT table_name FROM `4_prod`.information_schema.tables
        WHERE table_schema = 'gold' AND table_type IN ('MATERIALIZED_VIEW', 'VIEW')
          AND table_name RLIKE '^(clinical|spine|reference|text)_'
    """).collect()
)


def gold_cohort_filter(table, table_metadata):
    """Return (WHERE clause, description) restricting one gold table to the cohort, or (None, ...) for full tables."""
    if "person_id" in table_metadata:
        return f"g.person_id IN ({cohort_person_ids()})", "person_id in cohort"
    if "parent_person_id" in table_metadata:
        return f"g.parent_person_id IN ({cohort_person_ids()})", "parent_person_id in cohort"
    if table == "spine_person_relationship":
        # A relationship row is about the cohort member on either side of it. These two columns are STRING in gold.
        return (f"(g.source_person_id IN ({cohort_person_ids('STRING')}) "
                f"OR g.target_person_id IN ({cohort_person_ids('STRING')}))",
                "source_person_id or target_person_id in cohort")
    if table in GOLD_CHILD_LINKS:
        child_keys, parents = GOLD_CHILD_LINKS[table]
        parent_queries = " UNION ".join(
            f"SELECT {', '.join(keys)} FROM `4_prod`.gold.{parent} WHERE person_id IN ({cohort_person_ids()})"
            for parent, keys in parents
        )
        child_tuple = ", ".join(f"g.{k}" for k in child_keys)
        description = "key in cohort rows of " + " / ".join(f"{p}.{'+'.join(k)}" for p, k in parents)
        return f"({child_tuple}) IN ({parent_queries})", description
    if table in GOLD_FULL_TABLES:
        return None, "not patient-level: delivered in full"
    raise ValueError(
        f"4_prod.gold.{table} is not classified. Add it to GOLD_CHILD_LINKS, GOLD_FULL_TABLES or GOLD_EXCLUDED."
    )


gold_built = 0
for table in gold_tables:
    if table in GOLD_EXCLUDED:
        continue
    table_metadata = gold_metadata[table]
    where, description = gold_cohort_filter(table, table_metadata)
    columns = delivered_columns(table_metadata)
    view = f"gold_{table}"
    spark.sql(f"""
        CREATE OR REPLACE VIEW {TARGET}.`{view}` AS
        SELECT {select_list(columns, 'g')}
        FROM `4_prod`.gold.`{table}` g
        {'WHERE ' + where if where else ''}
    """)
    built_views.append(dict(dataset="gold", view=view, source=f"4_prod.gold.{table}", columns=columns,
                            extra_columns=[], cohort_filter=description))
    gold_built += 1

# Every class-2/3 entry must still exist in gold, or the lists above have drifted.
stale = (set(GOLD_CHILD_LINKS) | GOLD_FULL_TABLES | GOLD_EXCLUDED) - set(gold_tables)
assert not stale, f"Listed gold tables no longer exist: {sorted(stale)}"
print(f"Gold Journey: {gold_built} views ({len(GOLD_EXCLUDED)} excluded)")

## 7. Schema mapping
One table holds every delivered column, built from the metadata gathered above. Three views expose it per dataset,
and ADF delivers those views alongside the data. The table itself is not delivered.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

MAPPING_TABLE = f"{TARGET}.schema_mapping_source"


def mapping_rows(entry):
    """Yield one schema mapping row per delivered column of a built view, in delivered order."""
    position = 0
    for extra in entry["extra_columns"]:
        position += 1
        yield (entry["dataset"], entry["view"], position, extra["column_name"], extra["full_data_type"],
               extra["comment"], entry["source"], extra["source_column"], None, None, entry["cohort_filter"])
    for info in entry["columns"]:
        position += 1
        yield (entry["dataset"], entry["view"], position, info["column_name"], delivered_type(info),
               info["comment"], entry["source"], info["column_name"], info["ig_risk"], info["ig_severity"],
               entry["cohort_filter"])


mapping_schema = StructType([
    StructField("dataset", StringType()), StructField("delivered_table", StringType()),
    StructField("ordinal_position", IntegerType()), StructField("column_name", StringType()),
    StructField("data_type", StringType()), StructField("description", StringType()),
    StructField("source_table", StringType()), StructField("source_column", StringType()),
    StructField("ig_risk", StringType()), StructField("ig_severity", StringType()),
    StructField("cohort_restriction", StringType()),
])
mapping = spark.createDataFrame([row for entry in built_views for row in mapping_rows(entry)], mapping_schema)
mapping.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(MAPPING_TABLE)

for dataset in ("rde", "bloodcounts", "gold"):
    spark.sql(f"""
        CREATE OR REPLACE VIEW {TARGET}.schema_mapping_{dataset} AS
        SELECT delivered_table, ordinal_position, column_name, data_type, description,
               source_table, source_column, ig_risk, ig_severity, cohort_restriction
        FROM {MAPPING_TABLE}
        WHERE dataset = '{dataset}'
        ORDER BY delivered_table, ordinal_position
    """)
print(f"Schema mapping: {mapping.count():,} columns over {len(built_views)} views")

## 8. Checks
Cheap checks only: every view must resolve, and the delivered column lists must match the mapping.
Full row counts would scan everything ADF is about to export anyway, so they are skipped. The cohort coverage
figure is one aggregate over the person-level anchors.

In [0]:
views_now = views_in_target()
expected = {e["view"] for e in built_views} | {f"schema_mapping_{d}" for d in ("rde", "bloodcounts", "gold")}
assert views_now == expected, f"View set mismatch: missing {sorted(expected - views_now)}, unexpected {sorted(views_now - expected)}"

# Resolving each view's schema compiles it against its source without reading data.
for entry in built_views:
    delivered = spark.table(f"{TARGET}.`{entry['view']}`").columns
    mapped = [c["column_name"] for c in entry["extra_columns"]] + [c["column_name"] for c in entry["columns"]]
    assert delivered == mapped, f"{entry['view']}: delivered columns differ from the schema mapping"

# ADF Copy fails on nested Parquet columns, so no delivered view may have one.
nested_left = spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM `{target_catalog}`.information_schema.columns
    WHERE table_schema = '{target_schema}'
      AND upper(data_type) IN ('ARRAY', 'STRUCT', 'MAP', 'VARIANT')
""").collect()
assert not nested_left, f"Nested columns would break the ADF copy: {[tuple(r) for r in nested_left]}"

coverage = spark.sql(f"""
    SELECT
        (SELECT count(DISTINCT PERSON_ID) FROM {COHORT_TABLE} WHERE PERSON_ID IS NOT NULL)                AS cohort_persons,
        (SELECT count(DISTINCT person_id) FROM {TARGET}.gold_spine_person)                               AS in_gold_spine_person,
        (SELECT count(DISTINCT PERSON_ID) FROM {TARGET}.rde_patient_demographics)                        AS in_rde_demographics,
        (SELECT count(DISTINCT PERSON_ID) FROM {TARGET}.bloodcounts_service_in)                            AS in_bloodcounts
""").collect()[0].asDict()
for name, value in coverage.items():
    print(f"{name:>22}: {value:,}")

summary = {d: sum(1 for e in built_views if e["dataset"] == d) for d in ("rde", "bloodcounts", "gold")}
print(f"Built in {TARGET}: {summary} plus 3 schema mapping views. Ready for ADF delivery (clinical_schema = {TARGET}).")